# Notebook 03 — Mô phỏng Bài nộp Sinh viên
**Nhóm 67 | Tuần 2**

Mô phỏng 12 bài nộp có lỗi cố ý để đo FPR thực tế (RQ1).

> Chạy notebook 00 trước.

In [ ]:
import sys, os, json, csv

BASE = '/content/nhom67'
sys.path.insert(0, f'{BASE}/src')
from runner import grade_submission, compute_fpr

OUT_DIR = f'{BASE}/results'
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset
with open(f'{BASE}/data/processed/mbpp_clean.json', encoding='utf-8') as f:
    problems = {p['task_id']: p for p in json.load(f)}

print(f'✓ Đọc được {len(problems)} bài')

In [ ]:
# 12 bài nộp mô phỏng có lỗi cố ý
submissions = [
    {
        'task_id': 1, 'sv_id': 'SV001',
        'mo_ta_loi': 'Dùng vòng lặp đúng nhưng cộng index thay vì giá trị',
        'code': 'def sum_list(lst):\n    total = 0\n    for i in range(len(lst)):\n        total += i\n    return total\n'
    },
    {
        'task_id': 1, 'sv_id': 'SV002',
        'mo_ta_loi': 'Quên return — trả về None',
        'code': 'def sum_list(lst):\n    total = 0\n    for x in lst:\n        total += x\n'
    },
    {
        'task_id': 2, 'sv_id': 'SV003',
        'mo_ta_loi': 'Syntax Error — thiếu dấu :',
        'code': 'def is_even(n)\n    return n % 2 == 0\n'
    },
    {
        'task_id': 4, 'sv_id': 'SV004',
        'mo_ta_loi': 'sorted() lấy phần tử đầu thay vì cuối — pass 2/3 public',
        'code': 'def find_max(lst):\n    return sorted(lst)[0]\n'
    },
    {
        'task_id': 6, 'sv_id': 'SV005',
        'mo_ta_loi': 'Không xử lý n=0 — pass public (5,3,1) nhưng fail hidden (0)',
        'code': 'def factorial(n):\n    result = 1\n    for i in range(1, n + 1):\n        result *= i\n    return result\n'
    },
    {
        'task_id': 7, 'sv_id': 'SV006',
        'mo_ta_loi': 'So sánh sai — luôn trả True',
        'code': 'def is_palindrome(s):\n    return True\n'
    },
    {
        'task_id': 8, 'sv_id': 'SV007',
        'mo_ta_loi': 'Dùng set mất thứ tự — WA với hidden test thứ tự quan trọng',
        'code': 'def remove_duplicates(lst):\n    return list(set(lst))\n'
    },
    {
        'task_id': 9, 'sv_id': 'SV008',
        'mo_ta_loi': 'Không xử lý danh sách rỗng — ZeroDivisionError với hidden test []',
        'code': 'def average(lst):\n    return sum(lst) / len(lst)\n'
    },
    {
        'task_id': 11, 'sv_id': 'SV009',
        'mo_ta_loi': 'Không xử lý n=0, n=1 — pass public (7,4,2) fail hidden (0,1)',
        'code': 'def is_prime(n):\n    for i in range(2, n):\n        if n % i == 0:\n            return False\n    return True\n'
    },
    {
        'task_id': 12, 'sv_id': 'SV010',
        'mo_ta_loi': 'Tính tổng số lẻ thay vì số chẵn',
        'code': 'def sum_even(lst):\n    return sum(x for x in lst if x % 2 != 0)\n'
    },
    {
        'task_id': 13, 'sv_id': 'SV011',
        'mo_ta_loi': 'Đệ quy thiếu base case n=0 — RuntimeError với hidden test',
        'code': 'def fibonacci(n):\n    if n == 1:\n        return 1\n    return fibonacci(n-1) + fibonacci(n-2)\n'
    },
    {
        'task_id': 14, 'sv_id': 'SV012',
        'mo_ta_loi': 'Không xử lý chuỗi toàn khoảng trắng — WA với hidden test',
        'code': 'def count_words(s):\n    return len(s.split())\n'
    },
]

In [ ]:
print('=' * 70)
print(f'  MÔ PHỎNG {len(submissions)} BÀI NỘP SINH VIÊN')
print('=' * 70)

sim_rows = []
fp_count = 0

for sub in submissions:
    tid   = sub['task_id']
    sv_id = sub['sv_id']
    code  = sub['code']
    prob  = problems.get(tid)
    if not prob:
        print(f'  [!] task_id {tid} không tìm thấy trong dataset')
        continue

    func = [
        l.split('(')[0].replace('def ','').strip()
        for l in code.split('\n')
        if l.strip().startswith('def ')
    ]
    if not func:
        print(f'  [{sv_id}] Không tìm được tên hàm')
        continue
    func_name = func[0]

    pub_r = grade_submission(code, func_name, prob['public_tests'], 'public')
    hid_r = grade_submission(code, func_name, prob['hidden_tests'], 'hidden')
    fpr   = compute_fpr(pub_r, hid_r)

    if fpr['is_false_positive']:
        fp_count += 1
        fp_flag = ' ★ FALSE POSITIVE'
    else:
        fp_flag = ''

    print(f'[{sv_id}] Bài {tid} — {func_name}()')
    print(f'  Lỗi : {sub["mo_ta_loi"]}')
    print(f'  Public : {pub_r["pass_count"]}/{pub_r["total_count"]} ({pub_r["test_pass_rate"]}%)')
    print(f'  Hidden : {hid_r["pass_count"]}/{hid_r["total_count"]} ({hid_r["test_pass_rate"]}%){fp_flag}')
    print()

    sim_rows.append({
        'sv_id':             sv_id,
        'task_id':           tid,
        'func':              func_name,
        'mo_ta_loi':         sub['mo_ta_loi'],
        'pub_pass':          pub_r['pass_count'],
        'pub_total':         pub_r['total_count'],
        'pub_tpr':           pub_r['test_pass_rate'],
        'pub_errors':        str(pub_r['error_counts']),
        'hid_pass':          hid_r['pass_count'],
        'hid_total':         hid_r['total_count'],
        'hid_tpr':           hid_r['test_pass_rate'],
        'hid_errors':        str(hid_r['error_counts']),
        'is_false_positive': fpr['is_false_positive'],
    })

# Kết luận RQ1
total   = len(sim_rows)
fpr_pct = round(fp_count / total * 100, 2)

print('=' * 55)
print('  KẾT LUẬN RQ1')
print('=' * 55)
print(f'  Tổng bài nộp kiểm tra : {total}')
print(f'  False Positive (FP)    : {fp_count}/{total} bài')
print(f'  False Positive Rate    : {fpr_pct}%')
print()
print('  → Với 3 test/bài, hệ thống bỏ sót lỗi edge case.')
print('  → Thêm hidden test giúp phát hiện các lỗi này.')
print('=' * 55)

# Lưu
sim_json = {'tong_submissions': total, 'fp_count': fp_count,
            'fpr_rate_%': fpr_pct, 'submissions': sim_rows}

with open(f'{OUT_DIR}/student_simulation.json', 'w', encoding='utf-8') as f:
    json.dump(sim_json, f, ensure_ascii=False, indent=2)

with open(f'{OUT_DIR}/student_simulation.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(sim_rows[0].keys()))
    w.writeheader(); w.writerows(sim_rows)

print(f'\n✓ Lưu: {OUT_DIR}/student_simulation.json')
print(f'✓ Lưu: {OUT_DIR}/student_simulation.csv')